# Semantic Chunking

In [2]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

e:\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Initialize the model
model=SentenceTransformer("all-MiniLM-L6-v2")

# Sample text
text="""LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination."""

# split text into sentences
sentences = [s.strip() for s in text.split("\n") if s.strip()]

# Embed each sentence 
embedding = model.encode(sentences)

# Initialize the parameter
threshold = 0.7
chunks = []
current_chunk=[sentences[0]]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3063.62it/s]


In [4]:
# semantic grouping based on threshold
for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embedding[i-1]],
        [embedding[i]]
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]
chunks.append(" ".join(current_chunk))


In [6]:
print("Semantic Chunk")
for idx, chunk in enumerate(chunks):
    print(f"\nChunk {idx+1}:\n{chunk}")

Semantic Chunk

Chunk 1:
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

Chunk 2:
You can create chains, agents, memory, and retrievers.

Chunk 3:
The Eiffel Tower is located in Paris.

Chunk 4:
France is a popular tourist destination.


## RAG Pipeline Modular Coding

In [1]:
from sentence_transformers import SentenceTransformer
# Measure the cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings


In [30]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["OPEN_ROUTER_API_KEY"] = os.getenv("OPEN_ROUTER_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [15]:
## Custom Semantic chunker with threshold

class ThresholdSemanticChunker:
    def __init__(self, model_name="all-MiniLM-L6-v2",threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text: str):
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        embeddings = self.model.encode(sentences)

        chunks = []
        current_chunk = [sentences[0]]

        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    
    def split_documents(self,docs):
        result = []
        for doc in docs:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result

In [17]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [18]:
## Chunking
chunker = ThresholdSemanticChunker(threshold=0.7)
chunks=chunker.split_documents([doc])
chunks

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3638.93it/s]


[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.'),
 Document(metadata={}, page_content='You can create chains, agents, memory, and retrievers.'),
 Document(metadata={}, page_content='The Eiffel Tower is located in Paris.'),
 Document(metadata={}, page_content='France is a popular tourist destination.')]

In [21]:
## VectorStore

embeddings = OpenAIEmbeddings(
    api_key=os.getenv("OPEN_ROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
)

vector_store = FAISS.from_documents(
    chunks, embedding=embeddings
)

retriever = vector_store.as_retriever()


In [27]:
template = """
You are my personal AI assistant.

Use only the information provided in the context below to answer the question.
If the answer is not present in the context, say "I don't know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are my personal AI assistant.\n\nUse only the information provided in the context below to answer the question.\nIf the answer is not present in the context, say "I don\'t know based on the provided context."\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n')

In [41]:
llm = init_chat_model(model="openai/gpt-oss-120b",temperature=0.7, model_provider="groq")

In [45]:
from langchain_core.runnables import RunnableMap
rag_chain = (
    RunnableMap(
        {
            "context": lambda x: retriever.invoke(x["question"]),
            "question": lambda x: x["question"]
        }
    )
        | prompt
        | llm
        | StrOutputParser()
)

In [46]:
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

LangChain is a framework for building applications that use large language models (LLMs). It provides modular abstractions that let you combine LLMs with tools such as OpenAI and Pinecone.


# Semantic Chunker with Langchain

In [47]:
from langchain_openai import OpenAIEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import TextLoader

C:\Users\ARKAN\AppData\Local\Temp\ipykernel_24388\532246479.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


In [48]:
from dotenv import load_dotenv
import os

load_dotenv()

os.environ["OPEN_ROUTER_API_KEY"] = os.getenv("OPEN_ROUTER_API_KEY")

In [49]:
# load the document
loader = TextLoader("E:\RAG\Data\langchain_intro.txt")
docs = loader.load()

## Initialize embedding model 
embedding = OpenAIEmbeddings(
    api_key=os.getenv("OPEN_ROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    check_embedding_ctx_length=False,
)

## create the semantic chunker
chunker=SemanticChunker(embedding)

## splits the documents
chunks=chunker.split_documents(docs)

for i, chunk in enumerate(chunks):
    print(f"\nchunk {i+1}\n{chunk.page_content}")


chunk 1
LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.

chunk 2
You can create chains, agents, memory, and retrievers. The Eiffel Tower is located in Paris. France is a popular tourist destination.
